# EDA — Telco Customer Churn

Análise exploratória rápida antes da modelagem (portfólio / desafio técnico).

**Dataset:** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — arquivo `WA_Fn-UseC_-Telco-Customer-Churn.csv`.

## Objetivo
- Ver a **distribuição do alvo** (`Churn`).
- Comparar **quem cancela vs quem fica** em variáveis-chave (contrato, tempo de casa).

In [ ]:
import os
from itertools import islice
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

CSV_NAME = "WA_Fn-UseC_-Telco-Customer-Churn.csv"


def resolve_telco_csv() -> Path:
    """Localiza o CSV: env TELCO_CSV_PATH, ou sobe diretórios a partir do CWD até achar churn.py / arquivo."""
    env = os.environ.get("TELCO_CSV_PATH")
    if env:
        p = Path(env).expanduser().resolve()
        if p.is_file():
            return p
    here = Path.cwd().resolve()
    chain = [here, *list(islice(here.parents, 20))]
    for base in chain:
        for rel in (CSV_NAME, Path("notebooks") / CSV_NAME):
            cand = (base / rel).resolve()
            if cand.is_file():
                return cand
    return None


csv_path = resolve_telco_csv()
if csv_path is None:
    raise FileNotFoundError(
        f"Não encontrei '{CSV_NAME}'.\n"
        "• Baixe em https://www.kaggle.com/datasets/blastchar/telco-customer-churn\n"
        "• Coloque na pasta do projeto (ao lado de churn.py) ou em notebooks/\n"
        f"• CWD atual do kernel: {Path.cwd().resolve()}\n"
        "• Opcional: variável TELCO_CSV_PATH = caminho completo do .csv "
        "(PowerShell: $env:TELCO_CSV_PATH='C:\\...\\arquivo.csv'; Linux/macOS: export TELCO_CSV_PATH=...)"
    )

df = pd.read_csv(csv_path)
print("Arquivo:", csv_path)
df.shape

In [ ]:
df.info()

In [ ]:
print("Contagem:")
print(df["Churn"].value_counts())
print("\nProporção (%):")
print((df["Churn"].value_counts(normalize=True) * 100).round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
taxa = (
    df.assign(churn_sim=(df["Churn"] == "Yes").astype(int))
    .groupby("Contract", observed=True)["churn_sim"]
    .mean()
    .sort_values(ascending=True)
)
taxa.plot(kind="barh", ax=ax, color="#c0392b")
ax.set_xlabel("Taxa de churn (média)")
ax.set_ylabel("Tipo de contrato")
ax.set_title("Churn por tipo de contrato")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=ax)
ax.set_title("Tempo de casa (tenure) vs Churn")
plt.tight_layout()
plt.show()

## Leitura rápida
- A base é **desbalanceada**: predominam clientes com `Churn = No`. Isso motiva **SMOTE só no treino** e foco em **recall** para a classe churn na modelagem.
- **Contrato mensal** (`Month-to-month`) associa-se a maior taxa de cancelamento do que contratos de um ou dois anos — coerente com as importâncias do XGBoost e com ações de retenção.
- **Tenure** tende a ser menor entre quem cancela, reforçando tempo de casa como fator de retenção.

**Próximo passo no repositório:** `churn.py` (limpeza, encoding, treino, exportação `joblib`) e `app.py` (Streamlit).